In [12]:
# Sam Brown
# sam_brown@mines.edu
# June 12 2025
# Neural net for regression, predicting total_delta

import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


df = pd.read_csv('averages_events_2011-13')
df = df.iloc[1:-1] # account for shift to get mins until feature

In [5]:
# Define Features and Target
X = df[[ 'tide_height', 'tide_change', 'mins_since']]
y = df['total_delta'].values.reshape(-1,1)

# Split to train and test
X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = .2, random_state = 42)

#Standardize
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_scaled = scaler.fit_transform(y_train)
y_test_scaled = scaler.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

In [76]:
# Neural net
class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()
        self.fc1 = nn.Linear(3,16)
        self.fc2 = nn.Linear(16,8)
        self.output = nn.Linear(8,1)

    def forward(self,x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.output(x) # Because we are regressing we dont need activation function
        return x




In [78]:
# Instantiate
model = Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = .01)


In [80]:
# Training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad() # Clears grad

    # Predictions and loss
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    #Backprop
    loss.backward()

    # Update params
    optimizer.step()
    
    if (epoch+1) % 20 == 0: # Print update to ensure no problems
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


Epoch [20/200], Loss: 0.4054
Epoch [40/200], Loss: 0.1319
Epoch [60/200], Loss: 0.0990
Epoch [80/200], Loss: 0.0925
Epoch [100/200], Loss: 0.0904
Epoch [120/200], Loss: 0.0891
Epoch [140/200], Loss: 0.0883
Epoch [160/200], Loss: 0.0877
Epoch [180/200], Loss: 0.0872
Epoch [200/200], Loss: 0.0867


In [42]:
# Eval
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor)
    y_pred = scaler.inverse_transform(y_pred_scaled.numpy()) # Undo transformations
    y_test_orig = scaler.inverse_transform(y_test_tensor.numpy())

# predictions vs actual
y_pred_flat = y_pred.flatten() # reshape to 1D
y_test_flat = y_test_orig.flatten()

for i in range(len(y_pred_flat)):
    print(f"Predicted: {y_pred_flat[i]:.2f}, Actual: {y_test_flat[i]:.2f}")


NameError: name 'model' is not defined

In [91]:
from sklearn.metrics import r2_score
r2 = r2_score(y_test_orig, y_pred)
print(r2)

0.9200652837753296
